# 第11章 画像データを用いた機械学習

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍の入力番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/11/
- 演習の解答: https://ml.kano.ac/solutions/11/

## 画像分類

### データの読み込みと確認

**入力 11.1**　MNISTサブセットの読み込み

In [ ]:
import numpy as np
from urllib.request import urlretrieve

# MNISTサブセット（1,000枚）のダウンロードと読み込み
url = "https://ml.kano.ac/chapters/data/mnist_1000_raw.npz"
urlretrieve(url, "mnist_1000_raw.npz")
data = np.load("mnist_1000_raw.npz")
X, y = data["X"], data["y"]

print(f"データの形状: {X.shape}")
print(f"ラベルの形状: {y.shape}")
print(f"クラス: {np.unique(y)}")

**入力 11.2**　先頭10枚の画像の表示

In [ ]:
import matplotlib.pyplot as plt

# 先頭10枚の画像を表示
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"ラベル: {y[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### データの分割と正規化

**入力 11.3**　データの分割と正規化

In [ ]:
from sklearn.model_selection import train_test_split

# 訓練データとテストデータの分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 画素値の正規化（0〜255 → 0〜1）
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"訓練データ: {X_train.shape}")
print(f"テストデータ: {X_test.shape}")

### 生の画素値を用いた分類

**入力 11.4**　生の画素値による分類の学習と評価

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ロジスティック回帰モデルの学習
model_pixel = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)
model_pixel.fit(X_train, y_train)

# テストデータでの予測と評価
y_pred_pixel = model_pixel.predict(X_test)
acc_pixel = accuracy_score(y_test, y_pred_pixel)
print(f"生の画素値による分類精度: {acc_pixel:.4f}")

### 誤分類の分析

**入力 11.5**　誤分類された画像の表示

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 誤分類されたインデックスを取得
misclassified = np.where(y_pred_pixel != y_test)[0]
print(f"誤分類数: {len(misclassified)} / {len(y_test)}")

# 誤分類された画像の一部を表示
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    idx = misclassified[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"正解: {y_test[idx]}, 予測: {y_pred_pixel[idx]}")
    ax.axis("off")
plt.suptitle("誤分類の例（生の画素）")
plt.tight_layout()
plt.show()

**入力 11.6**　混同行列の計算と表示

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 混同行列の計算と表示
cm = confusion_matrix(y_test, y_pred_pixel)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("予測")
plt.ylabel("正解")
plt.title("混同行列（生の画素）")
plt.tight_layout()
plt.show()

## HOG特徴量を用いた分類

### HOG特徴量の抽出

**入力 11.7**　HOG特徴量の抽出

In [ ]:
from skimage.feature import hog
import numpy as np

# 画像配列からHOG特徴量を抽出する
def extract_hog_features(images):
    hog_features = []
    for img in images:
        feature = hog(
            img.reshape(28, 28),
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )
        hog_features.append(feature)
    return np.array(hog_features)

# HOG特徴量の抽出
X_train_hog = extract_hog_features(X_train)
X_test_hog = extract_hog_features(X_test)

print(f"HOG特徴量の形状（訓練）: {X_train_hog.shape}")
print(f"HOG特徴量の形状（テスト）: {X_test_hog.shape}")

### ロジスティック回帰による分類

**入力 11.8**　HOG特徴量による分類の学習と評価

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ロジスティック回帰モデルの学習
model_hog = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)
model_hog.fit(X_train_hog, y_train)

# テストデータでの予測と評価
y_pred_hog = model_hog.predict(X_test_hog)
acc_hog = accuracy_score(y_test, y_pred_hog)
print("HOG特徴量による分類精度:",
      f"{acc_hog:.4f}")

### 結果の比較

**入力 11.9**　生の画素値とHOG特徴量の精度比較

In [ ]:
print(f"生の画素値による分類精度: {acc_pixel:.4f}")
print(f"HOG特徴量による分類精度: {acc_hog:.4f}")
diff = acc_hog - acc_pixel
print(f"精度の改善: {diff:.4f} ({diff * 100:.2f}ポイント)")

## 演習問題

### 演習 11-1: 生の画素値による分類

MNIST サブセット（1,000 枚）の生の画素値を、ロジスティック回帰で分類してください。

**タスク**：

1. MNIST サブセット（1,000 枚）を読み込む
2. 訓練 80%・テスト 20% に分割（`random_state=42`, `stratify=y`）し、画素値を 0〜1 に正規化する
3. `LogisticRegression(max_iter=1000, random_state=42)` で学習し、テスト精度を表示する

**データセット**： [data/mnist_1000_raw.npz](data/mnist_1000_raw.npz)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. データの読み込み
data = np.load('mnist_1000_raw.npz')
X = data['X']  # (1000, 784)
y = data['y']  # (1000,)

# 2. 訓練・テストに分割し、画素値を 0〜1 に正規化

# 3. ロジスティック回帰で学習し、テスト精度を表示

[解答例を見る](https://ml.kano.ac/solutions/11/#solution-11-1)

### 演習 11-2: HOG 特徴量による分類

MNIST サブセット（1,000 枚）から HOG 特徴量を抽出し、ロジスティック回帰で分類してください。

**タスク**：

1. MNIST サブセット（1,000 枚）を読み込み、演習 11-1 と同じ分割・正規化を行う
2. 訓練・テストそれぞれの画像から HOG 特徴量を抽出する（`orientations=9`, `pixels_per_cell=(4, 4)`, `cells_per_block=(2, 2)`）
3. `LogisticRegression(max_iter=1000, random_state=42)` で学習し、テスト精度を表示する
4. 演習 11-1 の生の画素値の結果と比較し、どの程度改善したかを確認する

**データセット**： [data/mnist_1000_raw.npz](data/mnist_1000_raw.npz)

In [ ]:
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. データの読み込みと分割・正規化（演習 11-1 と同じ）
data = np.load('mnist_1000_raw.npz')
X = data['X']
y = data['y']

# 2. HOG 特徴量の抽出

# 3. ロジスティック回帰で学習・精度を表示

# 4. 演習 11-1 の結果と比較

[解答例を見る](https://ml.kano.ac/solutions/11/#solution-11-2)

### 演習 11-3: HOG パラメータの調整

HOG 特徴量の設定を変えて、特徴量の次元数と分類精度がどのように変化するか確認してください。

**タスク**：

1. 演習 11-2 と同じデータ・分割を使う
2. `pixels_per_cell=(7, 7)`（`orientations=9`）に変更した場合の次元数とテスト精度を求める
3. `orientations=4`（`pixels_per_cell=(4, 4)`）に変更した場合の次元数とテスト精度を求める
4. 演習 11-2 の基準設定（`pixels_per_cell=(4, 4)`, `orientations=9`）の結果と比較し、考察する

[解答例を見る](https://ml.kano.ac/solutions/11/#solution-11-3)

### 演習 11-4: 同一特徴量でのモデル比較

演習 11-2 と同じ HOG 特徴量に対して、3 つのモデルで分類精度を比較してください。

**タスク**：

1. 演習 11-2 と同じデータ・分割・HOG 特徴量を使う
2. 以下の 3 モデルでテスト精度を計算する
    - `DecisionTreeClassifier(random_state=42)`
    - `LogisticRegression(max_iter=1000, random_state=42)`
    - `RandomForestClassifier(n_estimators=100, random_state=42)`
3. 結果を一覧表示し、どのモデルが HOG 特徴量と相性が良いか考察する

[解答例を見る](https://ml.kano.ac/solutions/11/#solution-11-4)

### 演習 11-5: CIFAR-10 のカラー画像分類

カラー画像のベンチマーク CIFAR-10 のサブセット（1,000 枚）に対して、本章と同じ手順の画像分類を適用し、手書き数字との違いを確認してください。

**タスク**：

1. CIFAR-10 サブセット（1,000 枚）を読み込む
2. 生の画素値（カラーのまま平坦化し 0〜1 に正規化）を訓練 80%・テスト 20% に分割（`random_state=42`, `stratify=y`）し、ロジスティック回帰で分類してテスト精度を求める
3. グレースケール変換後に HOG 特徴量（`orientations=9`, `pixels_per_cell=(4, 4)`, `cells_per_block=(2, 2)`）を抽出し、同様に分類してテスト精度を求める
4. MNIST の結果（演習 11-1・11-2）と比較し、精度が変わる理由を考察する

発展として、11.2 節で学んだ RGB ヒストグラム特徴量を組み合わせ、色の情報がどの程度効くかを確かめるのも良いでしょう。

**データセット**： [data/cifar_1000_color.npz](data/cifar_1000_color.npz)

In [ ]:
import numpy as np
from skimage.color import rgb2gray
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. データの読み込み
data = np.load('cifar_1000_color.npz')
X = data['X']  # (1000, 32, 32, 3)
y = data['y']  # (1000,)

# グレースケール変換の例
X_gray = np.array([rgb2gray(img) for img in X])

# 2. 生の画素値による分類

# 3. グレースケール → HOG 特徴量による分類

# 4. 結果の比較

[解答例を見る](https://ml.kano.ac/solutions/11/#solution-11-5)